# Citiz : extraction des stations d'autopartage (GBFS) vers le Lakehouse

**Objectif** : récupérer toutes les stations Citiz publiées sur transport.data.gouv.fr et les écrire dans des tables Delta du Lakehouse, pour croiser ensuite avec la population (carreaux INSEE 200 m) et les arrêts TC.

**Étapes**
1. Interroger l'API `GET /api/datasets` du PAN et garder les jeux Citiz.
2. Récupérer l'URL GBFS de chaque jeu (`gbfs.json`), puis la liste des flux publiés.
3. Lire `station_information`, `station_status`, `vehicle_types`.
4. Nettoyer, dédoublonner, écrire 4 tables : `citiz_flux`, `citiz_stations`, `citiz_station_status`, `citiz_vehicle_types`.

**Avant de lancer** : attacher un Lakehouse au notebook (panneau de gauche). Le notebook marche aussi hors Fabric (pandas seul, export CSV) si `spark` n'existe pas.

**Collecter de l'historique** : GBFS ne donne que l'état instantané. Planifier ce notebook toutes les 5 à 15 min (Run > Schedule) ajoute un instantané à `citiz_station_status` à chaque exécution : c'est la seule façon d'obtenir un proxy d'usage.

In [2]:
import re, json, time
from datetime import datetime, timezone
import requests
import pandas as pd

API_DATASETS = "https://transport.data.gouv.fr/api/datasets"
HEADERS = {"User-Agent": "hackathon-citiz-fabric/1.0"}
SNAPSHOT_TS = datetime.now(timezone.utc)

session = requests.Session()
session.headers.update(HEADERS)

def get_json(url, retries=3, timeout=30):
    """GET avec quelques relances ; renvoie None si la ressource reste injoignable."""
    for attempt in range(retries):
        try:
            r = session.get(url, timeout=timeout)
            r.raise_for_status()
            return r.json()
        except Exception as e:
            if attempt == retries - 1:
                print(f"  échec {url} : {e}")
                return None
            time.sleep(2 ** attempt)

## 1. Trouver les jeux Citiz dans l'API du PAN

L'API renvoie tous les jeux d'un coup (liste JSON). On garde ceux dont le titre, le slug ou le producteur contient « citiz », puis leurs ressources au format `gbfs`.
On privilégie `original_url` (URL du producteur) à `url` (redirection data.gouv.fr).

In [3]:
datasets = get_json(API_DATASETS, timeout=120)
print(f"{len(datasets)} jeux dans le PAN")

def is_citiz(d):
    pub = d.get("publisher") or {}
    txt = " ".join(str(x) for x in [d.get("title"), d.get("slug"), pub.get("name")] if x)
    return "citiz" in txt.lower()

rows = []
for d in filter(is_citiz, datasets):
    for res in d.get("resources", []):
        if (res.get("format") or "").lower() != "gbfs":
            continue
        rows.append({
            "dataset_id": d.get("id"),
            "dataset_slug": d.get("slug"),
            "dataset_title": d.get("title"),
            "publisher": (d.get("publisher") or {}).get("name"),
            "page_url": d.get("page_url"),
            "licence": d.get("licence"),
            "resource_title": res.get("title"),
            "gbfs_url": res.get("original_url") or res.get("url"),
            "is_available": res.get("is_available"),
        })

flux = pd.DataFrame(rows).drop_duplicates("gbfs_url").reset_index(drop=True)
# identifiant court du flux : "provider_12", sinon le slug
flux["source"] = flux.apply(
    lambda r: (lambda m: f"provider_{m.group(1)}" if m else (r.dataset_slug or f"flux_{r.name}"))(re.search(r"provider/(\d+)", r.gbfs_url or "")),
    axis=1)
print(f"{flux.dataset_slug.nunique()} jeux Citiz, {len(flux)} flux GBFS")
flux[["source", "dataset_title", "gbfs_url", "is_available"]]

802 jeux dans le PAN
15 jeux Citiz, 15 flux GBFS


,source,dataset_title,gbfs_url,is_available
0,provider_12,Autopartage Citiz Développement,https://backend.citiz.fr/public/provider/12/gb...,True
1,provider_5,Autopartage Citiz Auvergne Rhône Alpes (AURA),https://backend.citiz.fr/public/provider/5/gbf...,True
2,provider_1,Autopartage Citiz Grand Est,https://backend.citiz.fr/public/provider/1/gbf...,True
3,offre-et-disponibilite-temps-reel-du-service-c...,Autopartage Citiz Métropole de Lyon,https://data.grandlyon.com/files/rdata/lpa_mob...,True
4,provider_6,Autopartage Citiz Provence,https://backend.citiz.fr/public/provider/6/gbf...,True
5,provider_8,Autopartage Citiz Occitanie,https://backend.citiz.fr/public/provider/8/gbf...,True
6,provider_16,Autopartage Citiz Haut de France,https://backend.citiz.fr/public/provider/16/gb...,True
7,provider_15,Autopartage Citiz Nantes Métropole,https://backend.citiz.fr/public/provider/15/gb...,True
8,provider_4,Autopartage Citiz Gironde,https://backend.citiz.fr/public/provider/4/gbf...,True
9,provider_19,Autopartage Citiz Rennes Métropole,https://backend.citiz.fr/public/provider/19/gb...,True


## 2. Lire le `gbfs.json` de chaque flux

GBFS 3.0 : `data.feeds = [{name, url}]`. GBFS 2.x : `data.<langue>.feeds`. On gère les deux.

In [4]:
WANTED = ["station_information", "station_status", "vehicle_types", "system_information"]

def list_feeds(gbfs):
    data = (gbfs or {}).get("data", {})
    if "feeds" in data:                       # GBFS 3.0
        feeds = data["feeds"]
    else:                                     # GBFS 2.x : on prend la 1re langue (fr si dispo)
        lang = "fr" if "fr" in data else next(iter(data), None)
        feeds = data.get(lang, {}).get("feeds", []) if lang else []
    return {f["name"]: f["url"] for f in feeds}

feed_urls = {}
for r in flux.itertuples():
    g = get_json(r.gbfs_url)
    feeds = list_feeds(g)
    feed_urls[r.source] = feeds
    flux.loc[r.Index, "gbfs_version"] = (g or {}).get("version")
    flux.loc[r.Index, "feeds"] = ",".join(sorted(feeds))
    print(f"{r.source:<20} v{(g or {}).get('version')}  {sorted(feeds)}")

provider_12          v3.0  ['gbfs', 'gbfs_versions', 'geofencing_zones', 'station_information', 'station_status', 'system_information', 'system_pricing_plans', 'vehicle_status', 'vehicle_types']
provider_5           v3.0  ['gbfs', 'gbfs_versions', 'geofencing_zones', 'station_information', 'station_status', 'system_information', 'system_pricing_plans', 'vehicle_status', 'vehicle_types']
provider_1           v3.0  ['gbfs', 'gbfs_versions', 'geofencing_zones', 'station_information', 'station_status', 'system_information', 'system_pricing_plans', 'vehicle_status', 'vehicle_types']
offre-et-disponibilite-temps-reel-du-service-citiz-en-autopartage-metropole-de-lyon v3.0  ['gbfs', 'gbfs_versions', 'geofencing_zones', 'station_information', 'station_status', 'system_information', 'system_pricing_plans', 'vehicle_status', 'vehicle_types']
provider_6           v3.0  ['gbfs', 'gbfs_versions', 'geofencing_zones', 'station_information', 'station_status', 'system_information', 'system_pricing_plans

## 3. Lire stations, statuts et types de véhicules

Particularités GBFS 3.0 gérées ici :
- `name` est une liste `[{text, language}]` (texte simple en 2.x) ;
- `last_reported` est une date RFC 3339 (timestamp Unix en 2.x).

In [5]:
def text_of(v):
    """Champ localisé GBFS 3.0 -> texte (fr en priorité)."""
    if isinstance(v, list):
        fr = [x.get("text") for x in v if x.get("language", "").startswith("fr")]
        return fr[0] if fr else (v[0].get("text") if v else None)
    return v

def to_ts(v):
    if v is None:
        return pd.NaT
    if isinstance(v, (int, float)):
        return pd.to_datetime(v, unit="s", utc=True)
    return pd.to_datetime(v, utc=True, errors="coerce")

def read_feed(source, name, key):
    url = feed_urls.get(source, {}).get(name)
    if not url:
        return []
    j = get_json(url)
    return ((j or {}).get("data") or {}).get(key, [])

info_rows, status_rows, vt_rows, sys_rows = [], [], [], []
for src in flux.source:
    for s in read_feed(src, "station_information", "stations"):
        info_rows.append({
            "source": src,
            "station_id": str(s.get("station_id")),
            "name": text_of(s.get("name")),
            "short_name": text_of(s.get("short_name")),
            "lat": s.get("lat"), "lon": s.get("lon"),
            "address": s.get("address"),
            "post_code": s.get("post_code"),
            "city": text_of(s.get("city")),
            "capacity": s.get("capacity"),
            "is_virtual_station": s.get("is_virtual_station"),
            "vehicle_types_capacity": json.dumps(s.get("vehicle_types_capacity")) if s.get("vehicle_types_capacity") else None,
        })
    for s in read_feed(src, "station_status", "stations"):
        status_rows.append({
            "source": src,
            "station_id": str(s.get("station_id")),
            "num_vehicles_available": s.get("num_vehicles_available", s.get("num_bikes_available")),
            "num_vehicles_disabled": s.get("num_vehicles_disabled"),
            "num_docks_available": s.get("num_docks_available"),
            "is_installed": s.get("is_installed"),
            "is_renting": s.get("is_renting"),
            "is_returning": s.get("is_returning"),
            "last_reported": to_ts(s.get("last_reported")),
            "vehicle_types_available": json.dumps(s.get("vehicle_types_available")) if s.get("vehicle_types_available") else None,
            "snapshot_ts": SNAPSHOT_TS,
        })
    for v in read_feed(src, "vehicle_types", "vehicle_types"):
        vt_rows.append({
            "source": src,
            "vehicle_type_id": str(v.get("vehicle_type_id")),
            "name": text_of(v.get("name")),
            "make": text_of(v.get("make")), "model": text_of(v.get("model")),
            "form_factor": v.get("form_factor"),
            "propulsion_type": v.get("propulsion_type"),
            "rider_capacity": v.get("rider_capacity"),
            "max_range_meters": v.get("max_range_meters"),
        })

stations = pd.DataFrame(info_rows)
status = pd.DataFrame(status_rows, columns=["source", "station_id", "num_vehicles_available", "num_vehicles_disabled",
    "num_docks_available", "is_installed", "is_renting", "is_returning", "last_reported",
    "vehicle_types_available", "snapshot_ts"])
vehicle_types = pd.DataFrame(vt_rows)
print(f"stations : {len(stations)}  |  statuts : {len(status)}  |  types de véhicules : {len(vehicle_types)}")
stations.head()

stations : 1614  |  statuts : 1614  |  types de véhicules : 337


,source,station_id,name,short_name,lat,lon,address,post_code,city,capacity,is_virtual_station,vehicle_types_capacity
0,provider_12,43,Halles,None,47.392776,0.679685,34 Rue de la Grosse Tour,None,None,3,False,None
1,provider_12,44,Constantine-Commerce,None,47.395157,0.684126,NaN,None,None,2,False,None
2,provider_12,45,Cathédrale-Sicard,None,47.394527,0.693761,17 place François Sicard,None,None,2,False,None
3,provider_12,46,Gare,None,47.390404,0.694493,NaN,None,None,2,False,None
4,provider_12,47,Jean Jaurès (16 bd Beranger),None,47.389912,0.687188,NaN,None,None,2,False,None


## 4. Nettoyage et dédoublonnage

- On retire les stations sans coordonnées ou hors des bornes France + DROM.
- Certains flux se recoupent (AURA / Lyon, « Citiz Développement »). Une même station peut apparaître deux fois : on garde une ligne par position arrondie à ~10 m + nom, et on conserve la liste des flux sources.
- On ajoute un nombre de véhicules disponibles moyen à la station pour la carte.

In [6]:
stations["lat"] = pd.to_numeric(stations["lat"], errors="coerce")
stations["lon"] = pd.to_numeric(stations["lon"], errors="coerce")
n0 = len(stations)
# Métropole (Corse incluse) + DROM
in_metro = stations.lat.between(41, 51.5) & stations.lon.between(-5.5, 10)
DROM = [(15.8, 16.6, -61.9, -60.9),   # Guadeloupe
        (14.3, 14.9, -61.3, -60.8),   # Martinique
        (2.0, 6.0, -54.7, -51.5),     # Guyane
        (-21.5, -20.8, 55.1, 55.9),   # La Réunion
        (-13.1, -12.6, 44.9, 45.4)]   # Mayotte
in_drom = pd.Series(False, index=stations.index)
for la0, la1, lo0, lo1 in DROM:
    in_drom |= stations.lat.between(la0, la1) & stations.lon.between(lo0, lo1)
stations = stations[stations.lat.notna() & (in_metro | in_drom)].copy()
print(f"{n0 - len(stations)} stations retirées (coordonnées absentes ou aberrantes)")

stations["dedup_key"] = (stations.lat.round(4).astype(str) + "_" + stations.lon.round(4).astype(str)
                         + "_" + stations.name.fillna("").str.lower().str.strip())
sources_par_station = stations.groupby("dedup_key")["source"].agg(lambda s: ",".join(sorted(set(s))))
stations = stations.drop_duplicates("dedup_key").copy()
stations["sources"] = stations.dedup_key.map(sources_par_station)
stations["station_uid"] = stations.source + ":" + stations.station_id
stations["extracted_at"] = SNAPSHOT_TS

# dernier statut connu joint à la station (pratique pour la carte)
status["station_uid"] = status.source + ":" + status.station_id
stations = stations.merge(
    status[["station_uid", "num_vehicles_available", "is_renting"]], on="station_uid", how="left")

print(f"{len(stations)} stations Citiz uniques")
stations.groupby("source").size().sort_values(ascending=False)

0 stations retirées (coordonnées absentes ou aberrantes)
1614 stations Citiz uniques


source
provider_5                                                                             411
provider_1                                                                             290
offre-et-disponibilite-temps-reel-du-service-citiz-en-autopartage-metropole-de-lyon    200
provider_8                                                                             142
provider_4                                                                             100
provider_6                                                                              97
provider_16                                                                             80
provider_19                                                                             66
provider_12                                                                             64
provider_2                                                                              60
provider_7                                                                         

In [7]:
# Contrôles rapides
print("Stations sans capacité :", stations.capacity.isna().sum())
print("Véhicules disponibles maintenant :", int(status.num_vehicles_available.fillna(0).sum()))
if "propulsion_type" in vehicle_types:
    print(vehicle_types.groupby("propulsion_type").size())

Stations sans capacité : 0
Véhicules disponibles maintenant : 2583
propulsion_type
combustion    182
electric       58
human          10
hybrid         87
dtype: int64


## 5. Écriture dans le Lakehouse

- `citiz_flux`, `citiz_stations`, `citiz_vehicle_types` : remplacées à chaque exécution (référentiel).
- `citiz_station_status` : **ajout** (append) à chaque exécution, pour construire l'historique de disponibilité.

Les stations ont aussi une colonne `geometry_wkt` (POINT lon lat) prête pour Power BI / la jointure spatiale.
Hors Fabric (pas de `spark`), on écrit des CSV dans `./output/`.

In [8]:
stations["geometry_wkt"] = "POINT(" + stations.lon.astype(str) + " " + stations.lat.astype(str) + ")"

def to_lakehouse(df, table, mode):
    if df.empty:                              # Spark ne sait pas inférer le schéma d'une table vide
        print(f"{table} : rien à écrire")
        return
    df = df.copy()
    for c in df.columns:                      # colonnes objet mixtes -> texte, pour Spark
        if df[c].dtype == "object":
            df[c] = df[c].astype("string")
    try:
        spark  # noqa: F821  (défini automatiquement dans Fabric)
    except NameError:
        import os
        os.makedirs("output", exist_ok=True)
        path = f"output/{table}.csv"
        df.to_csv(path, index=False, mode="a" if mode == "append" else "w",
                  header=not (mode == "append" and os.path.exists(path)))
        print(f"{table} -> {path} ({len(df)} lignes)")
        return
    sdf = spark.createDataFrame(df)          # noqa: F821
    (sdf.write.format("delta").mode(mode)
        .option("mergeSchema", "true")
        .saveAsTable(table))
    print(f"{table} : {mode} de {len(df)} lignes")

to_lakehouse(flux, "citiz_flux", "overwrite")
to_lakehouse(stations.drop(columns=["dedup_key"]), "citiz_stations", "overwrite")
to_lakehouse(vehicle_types, "citiz_vehicle_types", "overwrite")
to_lakehouse(status, "citiz_station_status", "append")

citiz_flux -> output/citiz_flux.csv (15 lignes)
citiz_stations -> output/citiz_stations.csv (1614 lignes)
citiz_vehicle_types -> output/citiz_vehicle_types.csv (337 lignes)
citiz_station_status -> output/citiz_station_status.csv (1614 lignes)


## 6. Vérification

Requête de contrôle sur les tables écrites (Fabric uniquement).

In [9]:
try:
    display(spark.sql("""
        SELECT source, COUNT(*) AS nb_stations, SUM(capacity) AS capacite_totale
        FROM citiz_stations GROUP BY source ORDER BY nb_stations DESC
    """))
    display(spark.sql("""
        SELECT COUNT(DISTINCT snapshot_ts) AS nb_instantanes, MIN(snapshot_ts) AS debut, MAX(snapshot_ts) AS fin
        FROM citiz_station_status
    """))
except NameError:
    print("Pas de Spark : voir les CSV dans ./output/")

Pas de Spark : voir les CSV dans ./output/


## Suite

- **Accès à 5 min à pied** : rayon ~400 m autour de chaque station (ou isochrone OSM), croisé avec les carreaux INSEE 200 m pour compter les habitants desservis.
- **Zones candidates** : carreaux peuplés, bien desservis en TC mais sans station Citiz à moins de 400 m.
- **Proxy d'usage** : à partir de `citiz_station_status` accumulé, taux de temps où la station n'a plus de voiture disponible.